# OpenAlex Pipeline

Enriches IEEE VIS papers with OpenAlex data (authors, institutions, citations).

**Input**: `data/processed/dataset_with_clusters.csv` (from BERTopic notebook)

**Outputs** (in `data/processed/outputs/openalex_notebook_outputs/tables/`):
- `works_enriched.csv` - papers with OpenAlex metadata
- `authors.csv` - unique authors
- `authorships.csv` - paper-author relationships
- `institutions.csv` - unique institutions with geo
- `coauthor_edges.csv` - author collaboration network
- `institution_edges.csv` - institution collaboration network
- Aggregated tables: `sec2a_*.csv`, `sec2b_*.csv`, `sec2d_*.csv`, `sec3a_*.csv`, `sec3b_*.csv`

In [26]:
import os, re, json, time, hashlib, itertools
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from tqdm.auto import tqdm

# Optional: better fuzzy matching
try:
    from rapidfuzz.fuzz import token_set_ratio as _fuzz_ratio
except ImportError:
    _fuzz_ratio = None

# Country info
import pycountry
from countryinfo import CountryInfo

---
## Configuration

In [27]:
# Paths
INPUT_CSV = Path("../data/processed/dataset_with_clusters.csv")
TOPIC_MAPPING = Path("../data/processed/topic_macro_mapping_renamed.csv")
OUT_DIR = Path("../data/processed/outputs/openalex_notebook_outputs")
TABLES_DIR = OUT_DIR / "tables"
CACHE_WORKS = OUT_DIR / "cache_openalex_works"
CACHE_INST = OUT_DIR / "cache_openalex_institutions"

# API settings
MAILTO = "your.email@example.com"  # OpenAlex polite pool
MIN_INTERVAL = 0.12  # ~8 req/s
TITLE_THRESHOLD = 92.0  # fuzzy match threshold

# Create directories
for d in [TABLES_DIR, CACHE_WORKS, CACHE_INST]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_CSV}")
print(f"Output: {TABLES_DIR}")

Input: ../data/processed/dataset_with_clusters.csv
Output: ../data/processed/outputs/openalex_notebook_outputs/tables


---
## Helper Functions

In [28]:
DOI_RE = re.compile(r"(10\.\d{4,9}/[-._;()/:A-Z0-9]+)", re.IGNORECASE)

def normalize_doi(raw: Any) -> Optional[str]:
    """Extract and normalize DOI, return None for placeholders."""
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    s = re.sub(r"https?://(dx\.)?doi\.org/", "", s).replace("doi:", "").strip()
    m = DOI_RE.search(s)
    if not m:
        return None
    doi = m.group(1).lower()
    return None if doi.startswith("10.0000") else doi  # skip placeholders

def normalize_doi_keep_placeholder(raw: Any) -> Optional[str]:
    """Extract DOI including placeholders (used as work_id)."""
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    s = re.sub(r"https?://(dx\.)?doi\.org/", "", s).replace("doi:", "").strip()
    m = DOI_RE.search(s)
    return m.group(1).lower() if m else None

def normalize_author_name(name: str) -> str:
    """Normalize author name to Title Case."""
    n = re.sub(r"\s+", " ", (name or "").strip())
    if "," in n and len(n.split(",")) == 2:
        last, first = [p.strip() for p in n.split(",", 1)]
        if first and last:
            n = f"{first} {last}"
    return " ".join(w.capitalize() if not (w.isupper() and len(w) <= 3) else w 
                    for w in n.split())

def short_id(url: Optional[str]) -> Optional[str]:
    """Extract short ID from OpenAlex URL."""
    return url.replace("https://openalex.org/", "") if url else None

def local_author_id(name: str) -> str:
    """Generate deterministic local ID for author without OpenAlex ID."""
    h = hashlib.sha1(name.lower().encode()).hexdigest()[:16]
    return f"local:{h}"

def fuzzy_match(a: str, b: str) -> float:
    """Fuzzy string similarity (0-100)."""
    if _fuzz_ratio:
        return float(_fuzz_ratio(a, b))
    import difflib
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio() * 100

def country_name(cc: str) -> Optional[str]:
    """Country name from ISO code."""
    try:
        return pycountry.countries.get(alpha_2=cc).name
    except:
        return None

def world_region(cc: str) -> Optional[str]:
    """Continent/region from country code."""
    name = country_name(cc)
    if not name:
        return None
    try:
        return CountryInfo(name).info().get("region")
    except:
        return None

---
## OpenAlex Client

In [29]:
class OpenAlexClient:
    """OpenAlex API client with caching and rate limiting."""
    
    def __init__(self, mailto: str, min_interval: float = 0.12):
        self.mailto = mailto
        self.min_interval = min_interval
        self.session = requests.Session()
        self._last_call = 0.0
    
    def _wait(self):
        """Rate limiting."""
        dt = time.time() - self._last_call
        if dt < self.min_interval:
            time.sleep(self.min_interval - dt)
    
    def _cache_path(self, kind: str, key: str) -> Path:
        """Get cache file path."""
        h = hashlib.sha1(key.encode()).hexdigest()
        folder = CACHE_WORKS if kind == "work" else CACHE_INST
        return folder / f"{h}.json"
    
    def _get(self, url: str, params: Dict = None) -> requests.Response:
        """Make API request with rate limiting."""
        self._wait()
        params = dict(params or {})
        if self.mailto:
            params["mailto"] = self.mailto
        resp = self.session.get(url, params=params, timeout=30)
        self._last_call = time.time()
        return resp
    
    def get_work_by_doi(self, doi: str) -> Tuple[Optional[Dict], str]:
        """Fetch work by DOI."""
        if not doi:
            return None, "no_doi"
        
        # Check cache
        cache = self._cache_path("work", f"doi:{doi}")
        if cache.exists():
            return json.loads(cache.read_text()), "cache"
        
        # API request
        url = f"https://api.openalex.org/works/https://doi.org/{doi}"
        for attempt in range(5):
            try:
                r = self._get(url)
                if r.status_code == 200:
                    data = r.json()
                    cache.write_text(json.dumps(data))
                    return data, "ok"
                if r.status_code == 404:
                    return None, "not_found"
                time.sleep(0.5 * (attempt + 1))
            except:
                time.sleep(0.5 * (attempt + 1))
        return None, "error"
    
    def search_work(self, title: str, year: int) -> Tuple[List[Dict], str]:
        """Search work by title and year."""
        if not title or not year:
            return [], "bad_args"
        
        cache = self._cache_path("work", f"search:{year}:{title}")
        if cache.exists():
            return json.loads(cache.read_text()), "cache"
        
        url = "https://api.openalex.org/works"
        params = {"search": title, "filter": f"publication_year:{year}", "per_page": 10}
        for attempt in range(5):
            try:
                r = self._get(url, params)
                if r.status_code == 200:
                    results = r.json().get("results", [])
                    cache.write_text(json.dumps(results))
                    return results, "ok"
                time.sleep(0.5 * (attempt + 1))
            except:
                time.sleep(0.5 * (attempt + 1))
        return [], "error"
    
    def get_institution(self, inst_id: str) -> Tuple[Optional[Dict], str]:
        """Fetch institution by ID."""
        inst_id = short_id(inst_id) or inst_id
        if not inst_id.startswith("I"):
            return None, "bad_id"
        
        cache = self._cache_path("inst", inst_id)
        if cache.exists():
            return json.loads(cache.read_text()), "cache"
        
        url = f"https://api.openalex.org/institutions/{inst_id}"
        for attempt in range(5):
            try:
                r = self._get(url)
                if r.status_code == 200:
                    data = r.json()
                    cache.write_text(json.dumps(data))
                    return data, "ok"
                if r.status_code == 404:
                    return None, "not_found"
                time.sleep(0.5 * (attempt + 1))
            except:
                time.sleep(0.5 * (attempt + 1))
        return None, "error"

client = OpenAlexClient(mailto=MAILTO, min_interval=MIN_INTERVAL)
print("OpenAlex client ready")

OpenAlex client ready


---
## 1. Load Dataset

In [30]:
df = pd.read_csv(INPUT_CSV)

# Normalize columns
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df["doi_query"] = df["DOI"].apply(normalize_doi)  # for API lookup
df["doi_unique"] = df["DOI"].apply(normalize_doi_keep_placeholder)  # as work_id

# Generate work_id (DOI or hash)
def make_work_id(row, idx):
    if row["doi_unique"]:
        return row["doi_unique"]
    base = f"{row.get('Conference','')}//{row.get('Year','')}//{row.get('Title','')}"
    return f"no_doi:{hashlib.sha1(base.encode()).hexdigest()[:16]}"

df["work_id"] = [make_work_id(r, i) for i, r in df.iterrows()]

# Load topic mapping and add macro names
topic_map = pd.read_csv(TOPIC_MAPPING)[["macro_id", "macro_name"]].drop_duplicates()
df = df.merge(topic_map, on="macro_id", how="left")

print(f"Papers: {len(df)}")
print(f"With DOI: {df['doi_query'].notna().sum()}")

Papers: 3530
With DOI: 3530


---
## 2. Resolve Works & Extract Authors

In [31]:
# Storage
works_rows = []
authorships_rows = []
authors_map = {}  # author_id -> author info

def pick_best_match(title: str, candidates: List[Dict]) -> Optional[Dict]:
    """Find best matching work by title similarity."""
    best, best_score = None, 0
    for c in candidates:
        score = fuzzy_match(title, c.get("title", ""))
        if score > best_score:
            best, best_score = c, score
    return best if best_score >= TITLE_THRESHOLD else None

for idx, row in tqdm(df.iterrows(), total=len(df), desc="OpenAlex resolve"):
    title = str(row.get("Title", "")).strip()
    year = int(row["Year"]) if pd.notna(row.get("Year")) else None
    work_id = row["work_id"]
    doi_query = row.get("doi_query")
    
    openalex_work = None
    matched_by = "none"
    
    # 1. Try DOI lookup
    if doi_query:
        openalex_work, _ = client.get_work_by_doi(doi_query)
        if openalex_work:
            matched_by = "doi"
    
    # 2. Try title+year search
    if not openalex_work and title and year:
        candidates, _ = client.search_work(title, year)
        best = pick_best_match(title, candidates)
        if best:
            # Fetch full record
            wid = best.get("id", "").replace("https://openalex.org/", "")
            url = f"https://api.openalex.org/works/{wid}"
            try:
                r = client._get(url)
                if r.status_code == 200:
                    openalex_work = r.json()
                    matched_by = "title_year"
            except:
                pass
    
    # Build works row
    w = {c: row[c] for c in df.columns}
    w["matched_by"] = matched_by
    if openalex_work:
        w["openalex_id"] = short_id(openalex_work.get("id"))
        w["openalex_citations"] = openalex_work.get("cited_by_count")
    else:
        w["openalex_id"] = None
        w["openalex_citations"] = None
    works_rows.append(w)
    
    # Extract authorships
    if openalex_work:
        for pos, auth in enumerate(openalex_work.get("authorships", []), 1):
            author = auth.get("author", {})
            author_id = short_id(author.get("id")) or local_author_id(
                normalize_author_name(author.get("display_name", f"unknown_{pos}"))
            )
            name = normalize_author_name(author.get("display_name", ""))
            
            # Upsert author
            if author_id not in authors_map:
                authors_map[author_id] = {
                    "author_id": author_id,
                    "author_name": name,
                    "orcid": author.get("orcid"),
                    "source": "openalex" if author_id.startswith("A") else "local"
                }
            
            # Institutions
            insts = auth.get("institutions", [])
            inst_ids = [short_id(i.get("id")) for i in insts if i.get("id")]
            
            authorships_rows.append({
                "work_id": work_id,
                "author_id": author_id,
                "author_position": pos,
                "institutions": json.dumps(inst_ids),
                "from_openalex": True
            })
    else:
        # Fallback: use dataset authors
        raw_names = str(row.get("AuthorNames-Deduped", "") or row.get("AuthorNames", "")).split(";")
        for pos, name in enumerate([n.strip() for n in raw_names if n.strip()], 1):
            name = normalize_author_name(name)
            author_id = local_author_id(name)
            
            if author_id not in authors_map:
                authors_map[author_id] = {
                    "author_id": author_id,
                    "author_name": name,
                    "orcid": None,
                    "source": "local"
                }
            
            authorships_rows.append({
                "work_id": work_id,
                "author_id": author_id,
                "author_position": pos,
                "institutions": json.dumps([]),
                "from_openalex": False
            })

# Create DataFrames
works_df = pd.DataFrame(works_rows)
authorships_df = pd.DataFrame(authorships_rows)
authors_df = pd.DataFrame(list(authors_map.values()))

print(f"Works: {len(works_df)}")
print(f"Authors: {len(authors_df)}")
print(f"Authorships: {len(authorships_df)}")

OpenAlex resolve:   0%|          | 0/3530 [00:00<?, ?it/s]

Works: 3530
Authors: 7219
Authorships: 14002


---
## 3. Fetch Institutions

In [32]:
# Get unique institution IDs
def explode_institutions(df):
    tmp = df.copy()
    tmp["inst_ids"] = tmp["institutions"].apply(lambda x: json.loads(x) if x else [])
    tmp = tmp.explode("inst_ids")
    return tmp[tmp["inst_ids"].notna() & tmp["inst_ids"].str.startswith("I")]

inst_long = explode_institutions(authorships_df)
unique_inst_ids = sorted(inst_long["inst_ids"].unique())
print(f"Unique institutions: {len(unique_inst_ids)}")

# Fetch institution details
inst_rows = []
for iid in tqdm(unique_inst_ids, desc="Fetch institutions"):
    inst, _ = client.get_institution(iid)
    if not inst:
        continue
    
    geo = inst.get("geo", {})
    cc = inst.get("country_code")
    
    inst_rows.append({
        "institution_id": short_id(inst.get("id")),
        "institution_name": inst.get("display_name"),
        "country_code": cc,
        "country_name": country_name(cc),
        "world_region": world_region(cc),
        "latitude": geo.get("latitude"),
        "longitude": geo.get("longitude"),
        "city": geo.get("city")
    })

institutions_df = pd.DataFrame(inst_rows).drop_duplicates(subset=["institution_id"])
print(f"Institutions: {len(institutions_df)}")

Unique institutions: 1264


Fetch institutions:   0%|          | 0/1264 [00:00<?, ?it/s]

Institutions: 1262


---
## 4. Build Collaboration Networks

In [33]:
# Coauthor network (author ↔ author)
coauthor_edges = {}
for work_id, g in authorships_df.groupby("work_id"):
    aids = sorted(set(g["author_id"].dropna()))
    if len(aids) < 2:
        continue
    for a, b in itertools.combinations(aids, 2):
        key = (a, b)
        coauthor_edges[key] = coauthor_edges.get(key, 0) + 1

coauthor_df = pd.DataFrame([
    {"author_a": a, "author_b": b, "weight": w}
    for (a, b), w in coauthor_edges.items()
]).sort_values("weight", ascending=False)

# Institution network (institution ↔ institution)
inst_long = inst_long.rename(columns={"inst_ids": "institution_id"})
work_insts = inst_long.groupby("work_id")["institution_id"].apply(lambda x: sorted(set(x))).to_dict()

inst_edges = {}
for work_id, insts in work_insts.items():
    if len(insts) < 2:
        continue
    for a, b in itertools.combinations(insts, 2):
        key = (a, b)
        inst_edges[key] = inst_edges.get(key, 0) + 1

inst_edges_df = pd.DataFrame([
    {"institution_a": a, "institution_b": b, "weight": w}
    for (a, b), w in inst_edges.items()
]).sort_values("weight", ascending=False)

print(f"Coauthor edges: {len(coauthor_df)}")
print(f"Institution edges: {len(inst_edges_df)}")

Coauthor edges: 23784
Institution edges: 4448


---
## 5. Generate Aggregated Tables

In [34]:
# Merge work info for calculations
auth_work = authorships_df.merge(works_df[["work_id", "Year", "Award", "CitationCount_CrossRef", 
                                           "openalex_citations", "macro_name"]], 
                                  on="work_id", how="left")

# Pick best citation count
auth_work["citations"] = auth_work["openalex_citations"].fillna(auth_work["CitationCount_CrossRef"]).fillna(0)
auth_work["has_award"] = auth_work["Award"].notna() & (auth_work["Award"].astype(str).str.strip() != "")

In [35]:
# SEC 2a: Authors per paper by year
n_authors = authorships_df.groupby("work_id").size().reset_index(name="n_authors")
work_year = works_df[["work_id", "Year"]].merge(n_authors, on="work_id")

sec2a = (work_year.dropna(subset=["Year"])
         .groupby("Year")["n_authors"]
         .agg(["mean", "std", "min", "max", "count"])
         .reset_index()
         .rename(columns={"mean": "avg_authors", "std": "std_authors", 
                          "min": "min_authors", "max": "max_authors", "count": "n_papers"}))

sec2a.to_csv(TABLES_DIR / "sec2a_authors_per_paper_by_year.csv", index=False)
print(f"✓ sec2a_authors_per_paper_by_year.csv")

✓ sec2a_authors_per_paper_by_year.csv


In [36]:
# SEC 2b: Unique authors cumulative
auth_year = auth_work.dropna(subset=["Year"])
first_year = auth_year.groupby("author_id")["Year"].min().reset_index(name="first_year")
new_per_year = first_year.groupby("first_year").size().reset_index(name="new_authors")
new_per_year = new_per_year.rename(columns={"first_year": "Year"})
new_per_year["cumulative_unique_authors"] = new_per_year["new_authors"].cumsum()

new_per_year.to_csv(TABLES_DIR / "sec2b_unique_authors_cumulative.csv", index=False)
print(f"✓ sec2b_unique_authors_cumulative.csv")

✓ sec2b_unique_authors_cumulative.csv


In [37]:
# SEC 2d: Author stats (for bubble chart)
author_stats = (auth_work.groupby("author_id")
                .agg(papers=("work_id", "nunique"),
                     citations=("citations", "sum"),
                     awards=("has_award", "sum"))
                .reset_index())

author_stats = author_stats.merge(authors_df[["author_id", "author_name"]], on="author_id", how="left")

author_stats.to_csv(TABLES_DIR / "sec2d_author_stats.csv", index=False)
print(f"✓ sec2d_author_stats.csv | {len(author_stats)} authors")

✓ sec2d_author_stats.csv | 7219 authors


In [38]:
# SEC 3a: Institutions map
inst_papers = inst_long.groupby("institution_id")["work_id"].nunique().reset_index(name="paper_count")
sec3a = institutions_df.merge(inst_papers, on="institution_id", how="left")

sec3a.to_csv(TABLES_DIR / "sec3a_institutions_map.csv", index=False)
print(f"✓ sec3a_institutions_map.csv | {len(sec3a)} institutions")

✓ sec3a_institutions_map.csv | 1262 institutions


In [39]:
# SEC 3b: Institution × MacroCategory (for Sankey)
inst_work = inst_long.merge(works_df[["work_id", "macro_name"]], on="work_id", how="left")
sec3b = (inst_work.dropna(subset=["macro_name"])
         .groupby(["institution_id", "macro_name"])
         .agg(count=("work_id", "nunique"))
         .reset_index())

# Add institution names
sec3b = sec3b.merge(institutions_df[["institution_id", "institution_name"]], on="institution_id", how="left")
sec3b = sec3b.rename(columns={"macro_name": "macro_category"})

sec3b.to_csv(TABLES_DIR / "sec3b_institution_macrocategory_counts.csv", index=False)
print(f"✓ sec3b_institution_macrocategory_counts.csv")

✓ sec3b_institution_macrocategory_counts.csv


---
## 6. Save All Tables

In [40]:
# Save main tables
works_df.to_csv(TABLES_DIR / "works_enriched.csv", index=False)
authors_df.to_csv(TABLES_DIR / "authors.csv", index=False)
authorships_df.to_csv(TABLES_DIR / "authorships.csv", index=False)
institutions_df.to_csv(TABLES_DIR / "institutions.csv", index=False)
coauthor_df.to_csv(TABLES_DIR / "coauthor_edges.csv", index=False)
inst_edges_df.to_csv(TABLES_DIR / "institution_edges.csv", index=False)

print("\n=== Output Summary ===")
print(f"Works:        {len(works_df):,}")
print(f"Authors:      {len(authors_df):,}")
print(f"Authorships:  {len(authorships_df):,}")
print(f"Institutions: {len(institutions_df):,}")
print(f"Coauthor edges:    {len(coauthor_df):,}")
print(f"Institution edges: {len(inst_edges_df):,}")


=== Output Summary ===
Works:        3,530
Authors:      7,219
Authorships:  14,002
Institutions: 1,262
Coauthor edges:    23,784
Institution edges: 4,448


---
## Summary

Generated tables:
```
data/processed/outputs/openalex_notebook_outputs/tables/
├── works_enriched.csv
├── authors.csv
├── authorships.csv
├── institutions.csv
├── coauthor_edges.csv
├── institution_edges.csv
├── sec2a_authors_per_paper_by_year.csv
├── sec2b_unique_authors_cumulative.csv
├── sec2d_author_stats.csv
├── sec3a_institutions_map.csv
└── sec3b_institution_macrocategory_counts.csv
```